# PS12 Walkthrough — Bayesian Network Flow

This notebook explains **how `assignment.py` works**, step by step.

**Problem:** reason under uncertainty about road accidents / signal failures using a common-cause Bayesian Network and **exact inference by joint enumeration**.

**End-to-end flow:**

```text
inputPS12.txt
      |
      v
 parse_input()          <- validate problem id, scenarios, CPT keys
      |
      v
 CommonCauseBN          <- store priors & conditionals
      |
      v
 build_joint()          <- 8-row joint table from factorization
      |
      v
 enumerate_prob()       <- reusable P(query | evidence)
 independence checks    <- marginal & conditional
      |
      v
 outputPS12.txt
```

> Submission artifact remains the single file `assignment.py`. This notebook is for learning / explanation only.


## 0. Setup — import the submission module

We reuse the real implementation so the notebook stays consistent with what you submit.


In [11]:
# Run this notebook from the Assignment-2/ folder (same directory as assignment.py).
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

# Import the graded module's building blocks
from assignment import (
    SCENARIO_1,
    SCENARIO_2,
    CommonCauseBN,
    are_conditionally_independent,
    are_marginally_independent,
    enumerate_prob,
    format_prob,
    generate_output,
    marginal,
    parse_input,
)

# Resolve paths relative to assignment.py (works even if the kernel cwd differs)
BASE = Path("assignment.py").resolve().parent
INPUT_PATH = BASE / "inputPS12.txt"
os.chdir(BASE)  # so local imports and relative paths stay consistent

print("Working directory:", os.getcwd())
print("Input file:", INPUT_PATH)


Working directory: /Users/n0k00km/Documents/School/Sem2/G557 - ACI/Assignment/ACI-Assignment/Assignment-2
Input file: /Users/n0k00km/Documents/School/Sem2/G557 - ACI/Assignment/ACI-Assignment/Assignment-2/inputPS12.txt


## 1. Domain model — common-cause structure

Both scenarios share the **same graph shape**: one hidden cause X with two effects Y and Z.

```text
Scenario 1                    Scenario 2
----------                    ----------
     A (Accident)                  S (Signal Failure)
    / \                           / \
   v   v                         v   v
  D     E                       C     R
(Delay) (Emergency)          (Camera) (Sensor)
```

**Factorization** (effects are conditionally independent given the cause):

`P(X, Y, Z) = P(X) * P(Y | X) * P(Z | X)`

| Fact | Meaning |
|------|---------|
| Y ⊥ Z \| X | Knowing the cause, one effect adds no info about the other |
| Y and Z are *not* marginally independent | Without the cause, effects are dependent (shared cause) |


## 2. Step A — Parse & validate the input file

`parse_input()` reads `inputPS12.txt` and enforces:

1. First line is `PS12`
2. Known scenario headers only
3. Required `KEY=value` probabilities in [0, 1]
4. No missing / duplicate / unknown keys

CPTs are **never hard-coded** in the inference logic — evaluation may use different numbers.


In [12]:
# Show the raw input the program will consume
with open(INPUT_PATH, encoding="utf-8") as f:
    print(f.read())


PS12
SCENARIO_1_ROAD_ACCIDENT
P_A=0.07
P_D_given_A=0.88
P_D_given_notA=0.25
P_E_given_A=0.82
P_E_given_notA=0.10
SCENARIO_2_TRAFFIC_SIGNAL_FAILURE
P_S=0.05
P_C_given_S=0.90
P_C_given_notS=0.12
P_R_given_S=0.87
P_R_given_notS=0.08



In [13]:
# Parse into structured CPT dictionaries (one per scenario)
scenarios = parse_input(INPUT_PATH)

print("Scenario 1 keys →", scenarios[SCENARIO_1])
print("Scenario 2 keys →", scenarios[SCENARIO_2])


Scenario 1 keys → {'P_A': 0.07, 'P_D_given_A': 0.88, 'P_D_given_notA': 0.25, 'P_E_given_A': 0.82, 'P_E_given_notA': 0.1}
Scenario 2 keys → {'P_S': 0.05, 'P_C_given_S': 0.9, 'P_C_given_notS': 0.12, 'P_R_given_S': 0.87, 'P_R_given_notS': 0.08}


## 3. Step B — Build the Bayesian Network object

`CommonCauseBN` stores:

- prior P(X = T)
- P(Y = T | X = T), P(Y = T | X = F)
- P(Z = T | X = T), P(Z = T | X = F)

Complements like P(X = F) = 1 - P(X) are computed when needed.


In [14]:
# Scenario 1: X=A (accident), Y=D (delay), Z=E (emergency call)
p1 = scenarios[SCENARIO_1]

bn1 = CommonCauseBN(
    p_x=p1["P_A"],
    p_y_given_x=p1["P_D_given_A"],
    p_y_given_not_x=p1["P_D_given_notA"],
    p_z_given_x=p1["P_E_given_A"],
    p_z_given_not_x=p1["P_E_given_notA"],
)

print("P(A)        =", bn1.p_x)
print("P(D | A)    =", bn1.p_y_given_x)
print("P(D | ¬A)   =", bn1.p_y_given_not_x)
print("P(E | A)    =", bn1.p_z_given_x)
print("P(E | ¬A)   =", bn1.p_z_given_not_x)


P(A)        = 0.07
P(D | A)    = 0.88
P(D | ¬A)   = 0.25
P(E | A)    = 0.82
P(E | ¬A)   = 0.1


## 4. Step C — Construct the joint probability table

For every binary assignment (x, y, z):

`P(x, y, z) = P(x) * P(y | x) * P(z | x)`

Loop order matches the sample output (**must** be T-before-F for each variable):

```text
for X in [T, F]:
  for Y in [T, F]:
    for Z in [T, F]:
```

Rows are stored in `JointProbabilityTable` (capacity 8; insert fails if full).


In [15]:
var_names_1 = ("A", "D", "E")
joint1 = bn1.build_joint()

# Display as a tidy table
rows = [
    {
        "A": "T" if a else "F",
        "D": "T" if d else "F",
        "E": "T" if e else "F",
        "P(A,D,E)": format_prob(p),
        "raw": p,
    }
    for a, d, e, p in joint1.rows()
]
df_joint = pd.DataFrame(rows)
display(df_joint[["A", "D", "E", "P(A,D,E)"]])

# Sanity check: a valid joint must sum to 1
total = sum(r["raw"] for r in rows)
print(f"Sum of joint = {total:.12f}  (should be 1.0)")


,A,D,E,"P(A,D,E)"
0,T,T,T,0.0505
1,T,T,F,0.0111
2,T,F,T,0.0069
3,T,F,F,0.0015
4,F,T,T,0.0233
5,F,T,F,0.2093
6,F,F,T,0.0698
7,F,F,F,0.6278


Sum of joint = 1.000000000000  (should be 1.0)


### Manual check of one cell

Example: P(A=T, D=T, E=T) = P(A) * P(D | A) * P(E | A) = 0.07 × 0.88 × 0.82 = 0.050512 → printed as **0.0505** (round half up to 4 d.p.).


In [16]:
manual_ttt = 0.07 * 0.88 * 0.82
print("Exact TTT:", manual_ttt)
print("Formatted:", format_prob(manual_ttt))
print("From joint:", format_prob(joint1.rows()[0][3]))


Exact TTT: 0.050512
Formatted: 0.0505
From joint: 0.0505


## 5. Step D — Exact inference by enumeration

This is the **reusable** core (not a separate formula per query).

`P(Q = q | e) = sum of joint rows matching (e and Q=q) / sum of joint rows matching e`

`enumerate_prob(joint, query_var, query_value, evidence, var_names)`:

1. Restrict joint rows that match evidence e
2. Of those, keep rows where the query variable equals q
3. Divide (normalize)

Same function answers P(A | D), P(A | E), P(A | D, E), and Scenario 2 queries.


In [17]:
# Walk through P(A | D=T) with explicit marginals
p_ad = marginal(joint1, {"A": True, "D": True}, var_names_1)  # numerator mass
p_d = marginal(joint1, {"D": True}, var_names_1)              # evidence mass
p_a_given_d_manual = p_ad / p_d

p_a_given_d = enumerate_prob(
    joint1, query_var="A", query_value=True,
    evidence={"D": True}, var_names=var_names_1,
)

print(f"P(A=T, D=T)     = {p_ad:.6f}")
print(f"P(D=T)          = {p_d:.6f}")
print(f"P(A|D) manual   = {format_prob(p_a_given_d_manual)}")
print(f"P(A|D) engine   = {format_prob(p_a_given_d)}")


P(A=T, D=T)     = 0.061600
P(D=T)          = 0.294100
P(A|D) manual   = 0.2095
P(A|D) engine   = 0.2095


In [18]:
# All required Scenario 1 posteriors via the SAME engine
queries_s1 = [
    ("P(A | D)", {"D": True}),
    ("P(A | E)", {"E": True}),
    ("P(A | D and E)", {"D": True, "E": True}),
]

for label, evidence in queries_s1:
    posterior = enumerate_prob(
        joint1, "A", True, evidence, var_names_1
    )
    print(f"{label} = {format_prob(posterior)}")

print()
print("Notice: more consistent evidence => higher posterior on Accident.")


P(A | D) = 0.2095
P(A | E) = 0.3816
P(A | D and E) = 0.6848

Notice: more consistent evidence => higher posterior on Accident.


## 6. Step E — Independence analysis

### Marginal independence (no evidence on A)

Check whether P(D=T, E=T) ≈ P(D=T) * P(E=T).

If **not** equal → Delay and Emergency Call are **dependent** (expected: shared cause).

### Conditional independence given A

For each value a in {T, F}, check:

`P(D=T, E=T | a) ≈ P(D=T | a) * P(E=T | a)`

If both hold → D ⊥ E | A (matches the missing D–E edge / d-separation).


In [19]:
# Marginal dependence numbers
p_d = marginal(joint1, {"D": True}, var_names_1)
p_e = marginal(joint1, {"E": True}, var_names_1)
p_de = marginal(joint1, {"D": True, "E": True}, var_names_1)

print(f"P(D)P(E) = {p_d * p_e:.6f}")
print(f"P(D,E)   = {p_de:.6f}")
print("Equal?   ", abs(p_d * p_e - p_de) < 1e-9)

marg_indep = are_marginally_independent(joint1, "D", "E", var_names_1)
cond_indep = are_conditionally_independent(joint1, "A", "D", "E", var_names_1)

print()
print(
    "Traffic Delay and Emergency Call are",
    "independent" if marg_indep else "not independent",
    "without evidence.",
)
print(
    "Traffic Delay and Emergency Call are",
    "conditionally independent" if cond_indep else "not conditionally independent",
    "given Road Accident.",
)


P(D)P(E) = 0.044233
P(D,E)   = 0.073762
Equal?    False

Traffic Delay and Emergency Call are not independent without evidence.
Traffic Delay and Emergency Call are conditionally independent given Road Accident.


## 7. Scenario 2 — same pipeline, different variable names

No new inference code: map S, C, R into the same `CommonCauseBN` + `enumerate_prob`.


In [20]:
p2 = scenarios[SCENARIO_2]

bn2 = CommonCauseBN(
    p_x=p2["P_S"],
    p_y_given_x=p2["P_C_given_S"],
    p_y_given_not_x=p2["P_C_given_notS"],
    p_z_given_x=p2["P_R_given_S"],
    p_z_given_not_x=p2["P_R_given_notS"],
)

var_names_2 = ("S", "C", "R")
joint2 = bn2.build_joint()

queries_s2 = [
    ("P(S | C)", {"C": True}),
    ("P(S | R)", {"R": True}),
    ("P(S | C and R)", {"C": True, "R": True}),
]

for label, evidence in queries_s2:
    posterior = enumerate_prob(
        joint2, "S", True, evidence, var_names_2
    )
    print(f"{label} = {format_prob(posterior)}")

marg2 = are_marginally_independent(joint2, "C", "R", var_names_2)
cond2 = are_conditionally_independent(joint2, "S", "C", "R", var_names_2)

print()
print(
    "Camera Alert and Sensor Alert are",
    "independent" if marg2 else "not independent",
    "without evidence.",
)
print(
    "Camera Alert and Sensor Alert are",
    "conditionally independent" if cond2 else "not conditionally independent",
    "given Traffic Signal Failure.",
)


P(S | C) = 0.2830
P(S | R) = 0.3640
P(S | C and R) = 0.8111

Camera Alert and Sensor Alert are not independent without evidence.
Camera Alert and Sensor Alert are conditionally independent given Traffic Signal Failure.


## 8. Step F — Full program output

`generate_output()` stitches Scenario 1 (joint + queries + independence text) and Scenario 2 (queries + independence text) into the exact `outputPS12.txt` format.

`main()` in `assignment.py` only:
1. resolves paths next to the script
2. calls `parse_input` → `generate_output`
3. writes `outputPS12.txt`
4. prints errors to stderr on failure (bad input)


In [21]:
# Produce the full submission-style output string
full_output = generate_output(scenarios)
print(full_output)

# Optional: compare to the file written by `python assignment.py`
output_path = BASE / "outputPS12.txt"
if output_path.is_file():
    on_disk = output_path.read_text(encoding="utf-8")
    print("Matches outputPS12.txt?", full_output == on_disk)
else:
    print("(outputPS12.txt not found yet — run: python assignment.py)")


Scenario 1 - Joint Probability Table
A	D	E	Probability
T	T	T	0.0505
T	T	F	0.0111
T	F	T	0.0069
T	F	F	0.0015
F	T	T	0.0233
F	T	F	0.2093
F	F	T	0.0698
F	F	F	0.6278
Scenario 1: Road Accident Prediction
P(A | D) = 0.2095
P(A | E) = 0.3816
P(A | D and E) = 0.6848
Traffic Delay and Emergency Call are not independent without evidence.
Traffic Delay and Emergency Call are conditionally independent given Road Accident.
Scenario 2: Traffic Signal Failure Detection
P(S | C) = 0.2830
P(S | R) = 0.3640
P(S | C and R) = 0.8111
Camera Alert and Sensor Alert are not independent without evidence.
Camera Alert and Sensor Alert are conditionally independent given Traffic Signal Failure.

Matches outputPS12.txt? True


## 9. Error-handling flow (parser)

Evaluation may use messy files. The parser fails fast with a clear message for:

| Problem | Example |
|---------|---------|
| Wrong problem id | `PS99` |
| Missing key | no `P_A=` |
| Duplicate key | `P_A` twice |
| Out-of-range prob | `P_A=1.5` |
| Non-numeric | `P_A=abc` |
| Unknown scenario header | typo in scenario name |


In [22]:
import tempfile
from assignment import InputParseError

bad_inputs = {
    "bad problem id": "PS99\n",
    "out of range": (
        "PS12\nSCENARIO_1_ROAD_ACCIDENT\nP_A=1.5\n"
        "P_D_given_A=0.88\nP_D_given_notA=0.25\n"
        "P_E_given_A=0.82\nP_E_given_notA=0.10\n"
        "SCENARIO_2_TRAFFIC_SIGNAL_FAILURE\n"
        "P_S=0.05\nP_C_given_S=0.90\nP_C_given_notS=0.12\n"
        "P_R_given_S=0.87\nP_R_given_notS=0.08\n"
    ),
}

for name, content in bad_inputs.items():
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "inputPS12.txt")
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
        try:
            parse_input(path)
            print(name, "→ unexpectedly succeeded")
        except InputParseError as err:
            print(f"{name} → InputParseError: {err}")


bad problem id → InputParseError: Invalid problem-set identifier 'PS99'; expected 'PS12'.
out of range → InputParseError: Invalid probability for P_A: 1.5 is outside [0, 1].


## 10. Flow summary (cheat sheet)

| Step | Function / class | Role |
|------|------------------|------|
| 1 | `parse_input` | File → CPT dicts + validation |
| 2 | `CommonCauseBN` | Hold factorization parameters |
| 3 | `build_joint` / `JointProbabilityTable` | Materialize P(X,Y,Z) |
| 4 | `enumerate_prob` | Any posterior P(Q \| e) |
| 5 | `are_marginally_independent` | Y ⊥ Z ? |
| 6 | `are_conditionally_independent` | Y ⊥ Z \| X ? |
| 7 | `generate_output` / `main` | Format & write `outputPS12.txt` |

**Design takeaway:** one common-cause engine serves both traffic scenarios; adding a new query is just another `enumerate_prob(...)` call, not a new Bayes formula.

---

*For submission: run `python assignment.py` (not this notebook). Export design content from `DESIGN_DRAFT.md` to PDF when `<GROUP_ID>` is known.*
